# GA4 Raw 数据质量检查

## tl;dr

批次 `9a2d8d7c-0f5b-4c57-a54d-98c59b9c5a27` 的质量运行 `3d6ec6a4-6192-4707-81d2-0c12b3c1d1e9` 已通过。11 条 BLOCKER 全部通过；23 条 purchase 缺交易号和 437 条重复交易号 purchase 为 WARNING，订单事实层需应用单独策略。

## Context & Methods

### Key Assumptions

- Raw 表的一行代表一个 GA4 事件。
- 所有时间按 CSV 导出时的 UTC 值处理。
- 此 notebook 只读取本机 MySQL；密码来自项目根目录的 `.env`。
- 可执行规则定义位于 `src/run_ga4_quality.py`，审计 SQL 位于 `sql/quality/01_ga4_raw_quality_checks.sql`。

In [1]:
import os
from pathlib import Path

import pymysql
from dotenv import load_dotenv

env_file = Path('.env') if Path('.env').is_file() else Path('../.env')
load_dotenv(env_file)
connection = pymysql.connect(
    host='127.0.0.1',
    port=int(os.environ.get('MYSQL_PORT', '3306')),
    user='root',
    password=os.environ['MYSQL_ROOT_PASSWORD'],
    charset='utf8mb4',
)
batch_id = '9a2d8d7c-0f5b-4c57-a54d-98c59b9c5a27'
run_id = '3d6ec6a4-6192-4707-81d2-0c12b3c1d1e9'

## Data

In [2]:
with connection.cursor() as cursor:
    cursor.execute('''
      SELECT COUNT(*), MIN(event_date), MAX(event_date)
      FROM insightops_raw.ga4_event_raw WHERE batch_id = %s
    ''', (batch_id,))
    raw_summary = cursor.fetchone()
raw_summary

(640055, datetime.date(2020, 11, 1), datetime.date(2020, 11, 14))

## Results

In [3]:
with connection.cursor() as cursor:
    cursor.execute('''
      SELECT rule_id, severity, failed_rows, failed_rate, outcome
      FROM insightops_ops.data_quality_result
      WHERE run_id = %s ORDER BY rule_id
    ''', (run_id,))
    quality_results = cursor.fetchall()
quality_results

(('B01_batch_succeeded', 'BLOCKER', 0, Decimal('0E-8'), 'PASSED'),
 ('B02_rows_present', 'BLOCKER', 0, Decimal('0E-8'), 'PASSED'),
 ('B03_date_within_batch_range', 'BLOCKER', 0, Decimal('0E-8'), 'PASSED'),
 ('B04_timestamp_matches_event_date', 'BLOCKER', 0, Decimal('0E-8'), 'PASSED'),
 ('B05_required_fields_populated', 'BLOCKER', 0, Decimal('0E-8'), 'PASSED'),
 ('B06_valid_device_category', 'BLOCKER', 0, Decimal('0E-8'), 'PASSED'),
 ('B07_valid_page_location', 'BLOCKER', 0, Decimal('0E-8'), 'PASSED'),
 ('B08_nonnegative_purchase_revenue', 'BLOCKER', 0, Decimal('0E-8'), 'PASSED'),
 ('B09_revenue_only_on_purchase', 'BLOCKER', 0, Decimal('0E-8'), 'PASSED'),
 ('B10_purchase_has_revenue', 'BLOCKER', 0, Decimal('0E-8'), 'PASSED'),
 ('B11_unique_source_row_number', 'BLOCKER', 0, Decimal('0E-8'), 'PASSED'),
 ('W01_unknown_acquisition', 'WARNING', 0, Decimal('0E-8'), 'PASSED'),
 ('W02_duplicate_purchase_transaction',
  'WARNING',
  437,
  Decimal('0.00068275'),
  'FAILED'),
 ('W03_purchase_miss

## Takeaways

事件 Staging 可使用该批次。订单指标必须基于后续规范订单表：排除缺失 `transaction_id` 的 purchase，并对重复交易号执行确定性去重。